In [0]:
dbutils.widgets.text("source_table", "", "Source Table Name")

In [0]:

def validate_table(table_name):
    df = spark.table(table_name)
    summary = []
    for col in df.columns:
        dtype = dict(df.dtypes)[col]
        unique_count = df.select(col).distinct().count()
        null_count = df.filter(df[col].isNull()).count()
        empty_count = df.filter((df[col] == "") | (df[col].isNull())).count() if dtype == 'string' else null_count
        min_val = df.agg({col: "min"}).collect()[0][0] if dtype != 'string' else None
        max_val = df.agg({col: "max"}).collect()[0][0] if dtype != 'string' else None
        summary.append({
            "column": col,
            "data_type": dtype,
            "unique_count": unique_count,
            "min": str(min_val) if min_val is not None else None,
            "max": str(max_val) if max_val is not None else None,
            "empty_count": empty_count,
            "null_count": null_count
        })
    result_df = spark.createDataFrame(summary)
    display(result_df)

source_table = dbutils.widgets.get("source_table")
validate_table(source_table)